# II. データ取得：Python (SunPy/Fido) による体系的アプローチ

## A. 観測対象日時とデータセットの定義

本解析の対象日時は、2022年6月13日03:12 UTです。この時刻を中心として、各機器のデータを取得します。
対象機器と視野範囲は以下の通りです。

SDO/AIA: 0-1.1 R 
s
​
  (EUV波長: 211Å, 193Å, 171Å)
MLSO Mk4: 1.1-2.2 R 
s
​
  (白色光、偏光輝度 pB)
SOHO/LASCO C2: 2.2-6 R 
s
​
  (白色光)

## B. SDO/AIA データの取得 (0-1.1$R_s$)

SDO/AIAデータは、JSOC (Joint Science Operations Center) から取得するのが一般的です。FidoはJSOCクライアントを内包しており、これを利用します 10。AIAデータは高時間分解能（例：aia.lev1_euv_12sシリーズで12秒ケイデンス）で提供されるため、指定時刻に非常に近い画像を取得できる可能性が高いです http://jsoc.stanford.edu/ajax/lookdata.html?ds=aia.lev1_euv_12s

http://jsoc2.stanford.edu/data/aia/synoptic/

AIA_analysis.ipynb

## C. SOHO/LASCO C2 データの取得 (2.2-6$R_s$)

SOHO/LASCO C2データは、VSO (Virtual Solar Observatory) を介して、または他のデータプロバイダからFidoを通じて取得できます。LASCO C2データはレベル0.5 (LZ) FITSファイルとして提供されることがあり、その場合、ファイル名がシーケンシャルな整数であるため、img_hdr.txtファイルが時刻同定に重要となります。

lasco_analysis.ipynb

## D. MLSO Mk4 データの取得 (1.1-2.2$R_s$)

MLSO Mk4データは、Kコロナメータの偏光輝度（pB）FITSファイルです 。Fidoを介したMk4データのクエリは、地上観測であり、a.Instrument.mk4のような専用属性がない可能性があるため、最も注意が必要です。a.Provider（例：VSO経由のHAOまたはNSO）とa.Source（MLSO）を使用するか、sunpy.map.sources.KCorMap  がそのタグ付け方法を示唆している場合はa.Instrument('KCOR')を使用する必要があるかもしれません。VSOにおけるMLSO Mk4のプロバイダとしてHAOが示唆されています 。sunpy.map.sources.KCorMapのドキュメント  は、K-CorがMk4を置き換えたと述べていますが、データは依然としてMk4またはMLSOの一般的なKコロナメータのタグで見つかる可能性があります。 

https://www2.hao.ucar.edu/mlso

In [3]:
# # このクエリは、直接的なMk4属性がない可能性があるため、より推測的
# # オプション1: MLSOの一般的なコロナグラフまたはKCOR装置で試す
# mk4_query_option1 = Fido.search(
#     time_window_coronagraph, # 同様の広い時間窓を使用
#     a.Provider.vso, # またはNSOのような他のVSOプロバイダ (SDACはLASCOなど宇宙機が主)
#                   # HAOがVSOプロバイダリストにあればそれが適切
#     a.Source.mlso, # マウナロア太陽観測所
#     a.Instrument.mk4 # 存在しない可能性あり。'KCOR'または類似のものを試す
#     # 波長は'white-light'または指定なしの可能性
# )
# print(mk4_query_option1)

# # オプション2: 直接クエリが失敗した場合、MLSO/HAOアーカイブから手動でダウンロードし、
# # ローカルにロードするようユーザーを誘導。MLSOはデータを提供している [6, 7, 19]。
# # FITSファイルが期待される [6, 18, 20]。
# # mk4_files = Fido.fetch(mk4_query_option1)

# III. 機器別データ較正と準備：詳細な検討

## A. SDO/AIA 処理：EUV 基礎画像の構築

AIA_analysis.ipynb

## B. SOHO/LASCO C2 処理：外部コロナの解明

### 1. sunpy.map.LASCOMap への読み込み

### 2. 重要な背景光除去

コロナグラフデータにとって背景光除去は最も重要な処理ステップです。LASCO画像には、Fコロナ（黄道光）や装置の迷光が含まれており、これらがKコロナ（電子散乱による真の太陽コロナ）の信号を覆い隠す可能性があります。

手法:
差分法 (Running/Base differencing): CMEのような突発現象には有効ですが、静穏な背景光には不向きです 。   
月間最小背景光 (Monthly minimum background): 一般的なアプローチです 。各ピクセルについて長期間（例：1ヶ月）の最小強度から参照背景を作成します。   
SiRGraF (Simple Radial Gradient Filter): 最小背景を減算し、均一強度勾配画像で除算します 。   
NRGF (Normalizing Radial Gradient Filter): 別の動径フィルターです 。   


### 3. 正確なオカルターマスキング

In [ ]:
from astropy.coordinates import SkyCoord
# [55]からの改作
# pixel_coords = all_coordinates_from_map(lasco_map_processed_background)
# solar_center = SkyCoord(0*u.deg, 0*u.deg, frame=lasco_map_processed_background.coordinate_frame)
# pixel_radii = np.sqrt((pixel_coords.Tx - solar_center.Tx)**2 + \
#                       (pixel_coords.Ty - solar_center.Ty)**2)
# inner_mask_radius_rsun = 2.2 # ユーザー指定の内部エッジ
# mask_inner = pixel_radii < lasco_map_processed_background.rsun_obs * inner_mask_radius_rsun
# final_mask = mask_inner # 外部エッジは視野で処理されると仮定
# masked_lasco = Map(lasco_map_processed_background.data,
#                    lasco_map_processed_background.meta,
#                    mask=final_mask)